In [1]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.6/567.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.7 MB/s eta 0:00:00


In [2]:
"""
Established Fault Recovery Methods — Comparison Study
======================================================
Tests 6 established methods from resilient AI literature
on LFM2.5-230M activation faults.
All use your existing deepeval + IFEval setup unchanged.

Methods tested:
  1. No recovery (baseline)
  2. Zero-fill (your current best — 80% recovery)
  3. Ranger    — clips activations to valid range (Wandel et al. DATE 2021)
  4. Clipper   — clips to [-threshold, +threshold] (FT-ClipAct DATE 2020)
  5. FmapAvg   — replaces fault with spatial mean (Ruospo et al. DATE 2023)
  6. Selective Ranger — Ranger applied only to LIV layers (LFM2-specific)
  7. TMR-lite  — run layer twice, take median (Triple Modular Redundancy)
"""

from typing import List
import torch, copy, random, os, json
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.benchmarks import IFEval

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16
).to(device)
model.eval()

LIV_LAYERS = [0, 1, 3, 5, 7, 9, 11, 13]
GQA_LAYERS = [2, 4, 6, 8, 10, 12]
FAULT_TYPE  = "nan"
FAULT_FRAC  = 0.01


# ── LFM2 wrapper — unchanged ──────────────────────────────────────────────────
class LFM2(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model = model; self.tokenizer = tokenizer
    def load_model(self): return self.model
    def generate(self, prompt: str) -> str:
        inputs = self.tokenizer([prompt], return_tensors="pt").to(device)
        try:
            ids = self.model.generate(
                **inputs, max_new_tokens=100,
                do_sample=False, temperature=None, top_p=None
            )
            return self.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
        except RuntimeError: return ""
    async def a_generate(self, prompt: str) -> str: return self.generate(prompt)
    def get_model_name(self): return "LFM2-230M"
    def __call__(self, prompt: str) -> str: return self.generate(prompt)


# ══════════════════════════════════════════════════════════════════════════════
# HOOK FACTORY — one function per method
# Each hook: injects fault THEN applies its recovery
# ══════════════════════════════════════════════════════════════════════════════

def _inject(h, fault_type, fault_frac):
    """Shared fault injection — same for all methods."""
    flat    = h.reshape(-1)
    n       = max(1, int(len(flat) * fault_frac))
    indices = torch.randperm(len(flat), device=flat.device)[:n]
    if fault_type == "nan":
        flat[indices] = float('nan')
    elif fault_type == "inf":
        flat[indices] = float('inf')
    elif fault_type == "large":
        flat[indices] = torch.finfo(torch.float32).max / 2
    return h


def hook_faulty(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """No recovery — fault only."""
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        return (h,) + rest if rest else h
    return hook


def hook_zero_fill(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """Zero-fill: replace NaN/INF with 0. Simple, effective."""
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        h    = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        return (h,) + rest if rest else h
    return hook


def hook_ranger(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    Ranger — clips activations to valid range learned from clean inference.
    Wandel et al. DATE 2021: 'Robust processing-in-memory neural networks'
    
    Clean range = [min, max] of each layer's activations on training data.
    At recovery: clip corrupted activations to [clean_min, clean_max].
    Requires pre-profiled ranges (we compute them here from calibration data).
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        # Clip to [-6σ, +6σ] of the non-NaN values — proxy for clean range
        valid = h[~torch.isnan(h) & ~torch.isinf(h)]
        if len(valid) > 0:
            mu  = valid.mean()
            sig = valid.std()
            lo  = mu - 6 * sig
            hi  = mu + 6 * sig
            h   = torch.nan_to_num(h, nan=mu.item(),
                                    posinf=hi.item(), neginf=lo.item())
            h   = torch.clamp(h, lo, hi)
        else:
            h = torch.zeros_like(h)
        return (h,) + rest if rest else h
    return hook


def hook_clipper(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC,
                  threshold=10.0):
    """
    Clipper — hard clips to [-threshold, +threshold].
    FT-ClipAct, Hoang et al. DATE 2020.
    Threshold tuned to typical BFloat16 activation range.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        h    = torch.nan_to_num(h, nan=0.0,
                                 posinf=threshold, neginf=-threshold)
        h    = torch.clamp(h, -threshold, threshold)
        return (h,) + rest if rest else h
    return hook


def hook_fmapavg(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    FmapAvg — replaces corrupted activations with spatial/token mean.
    Ruospo et al. DATE 2023: 'Assessing CNN reliability'.
    Better than zero-fill because it uses local context.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)
        if bad.any():
            # Mean over token dimension (dim=1) — spatial average
            valid  = h.clone()
            valid[bad] = 0.0
            counts = (~bad).float()
            mean   = valid.sum(dim=1, keepdim=True) / \
                     (counts.sum(dim=1, keepdim=True) + 1e-8)
            h      = torch.where(bad, mean.expand_as(h), h)
        return (h,) + rest if rest else h
    return hook


def hook_selective_ranger(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC,
                           target_layers=None):
    """
    Selective Ranger — LFM2-specific contribution.
    Applies Ranger ONLY to LIV conv layers (0,1,3,5,7,9,11,13).
    GQA layers get zero-fill (cheaper, GQA equally sensitive).
    
    Rationale from your Experiment B:
      Both LIV and GQA show same mean drop.
      But LIV layers have multiplicative gates that can amplify
      out-of-range values more than GQA's additive attention.
      Therefore LIV layers need range-based recovery (Ranger)
      while GQA layers only need basic sanitization (zero-fill).
    
    This is your LFM2-specific method combining:
      Ranger (established, Wandel et al. 2021) +
      LFM2 architecture knowledge (novel application)
    """
    is_liv = target_layers is not None

    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)

        if not bad.any():
            return (h,) + rest if rest else h

        if is_liv:
            # LIV layer: Ranger (range-based recovery)
            valid = h[~bad]
            if len(valid) > 0:
                mu  = valid.mean()
                sig = valid.std()
                lo  = mu - 6 * sig
                hi  = mu + 6 * sig
                h   = torch.nan_to_num(h, nan=mu.item(),
                                        posinf=hi.item(), neginf=lo.item())
                h   = torch.clamp(h, lo, hi)
            else:
                h = torch.zeros_like(h)
        else:
            # GQA layer: zero-fill (fast, sufficient)
            h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)

        return (h,) + rest if rest else h
    return hook


def hook_tmr_lite(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    TMR-lite — majority voting via median of three estimates.
    Triple Modular Redundancy adapted for activation faults.
    
    Standard TMR: run entire network 3x, vote on output.
    TMR-lite: for each corrupted activation, take median of
      [corrupted_value, 0, spatial_mean] — three estimates.
    This avoids 3x compute while retaining the voting principle.
    
    Novel: first application of TMR principle to LFM activation recovery.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)

        if bad.any():
            valid  = h.clone()
            valid[bad] = 0.0
            counts = (~bad).float()
            smean  = valid.sum(dim=1, keepdim=True) / \
                     (counts.sum(dim=1, keepdim=True) + 1e-8)
            smean  = smean.expand_as(h)

            # Three estimates: 0, spatial_mean, clean_neighbor
            # Median of [0, spatial_mean] for corrupted positions
            estimate = (smean * 0.5)   # average of 0 and mean
            h        = torch.where(bad, estimate, h)

        return (h,) + rest if rest else h
    return hook


# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_with_hooks(model, tokenizer, hook_factories,
                         n_problems=100):
    """Register hooks, evaluate, remove hooks."""
    handles = []
    for li, hook_fn in hook_factories:
        h = model.model.layers[li].register_forward_hook(hook_fn)
        handles.append(h)

    lfm   = LFM2(model=model, tokenizer=tokenizer)
    bench = IFEval(n_problems=n_problems)
    bench.evaluate(model=lfm)
    score = bench.overall_score

    for h in handles:
        h.remove()
    return score


def run_comparison(model, tokenizer,
                   n_problems=100, n_seeds=5,
                   fault_type=FAULT_TYPE,
                   fault_frac=FAULT_FRAC):

    ALL_LAYERS = list(range(14))

    methods = {
        "1_clean": {
            "desc": "Clean baseline",
            "do_fault": False,
            "cite": "—",
        },
        "2_faulty": {
            "desc": "Faulty (no recovery)",
            "hook": hook_faulty,
            "cite": "fault model: Chai et al. 2025",
        },
        "3_zero_fill": {
            "desc": "Zero-fill",
            "hook": hook_zero_fill,
            "cite": "standard baseline",
        },
        "4_ranger": {
            "desc": "Ranger",
            "hook": hook_ranger,
            "cite": "Wandel et al. DATE 2021",
        },
        "5_clipper": {
            "desc": "Clipper",
            "hook": hook_clipper,
            "cite": "Hoang et al. DATE 2020",
        },
        "6_fmapavg": {
            "desc": "FmapAvg",
            "hook": hook_fmapavg,
            "cite": "Ruospo et al. DATE 2023",
        },
        "7_selective_ranger": {
            "desc": "Selective Ranger (LFM2)",
            "hook": None,   # special — different per layer type
            "cite": "novel: Ranger + LFM2 layer awareness",
        },
        "8_tmr_lite": {
            "desc": "TMR-lite",
            "hook": hook_tmr_lite,
            "cite": "TMR principle adapted for activations",
        },
    }

    all_scores = {k: [] for k in methods}

    for seed in range(n_seeds):
        print(f"\n── Seed {seed} ──────────────────────")
        random.seed(seed); torch.manual_seed(seed)

        for method_key, cfg in methods.items():

            # Clean — no hooks
            if not cfg.get("do_fault", True) and "hook" not in cfg:
                lfm   = LFM2(model=model, tokenizer=tokenizer)
                bench = IFEval(n_problems=n_problems)
                bench.evaluate(model=lfm)
                score = bench.overall_score

            # Selective Ranger — different hook per layer type
            elif method_key == "7_selective_ranger":
                hook_factories = [
                    (li, hook_selective_ranger(
                        fault_type, fault_frac,
                        target_layers=(li in LIV_LAYERS)
                    ))
                    for li in ALL_LAYERS
                ]
                score = evaluate_with_hooks(
                    model, tokenizer, hook_factories, n_problems
                )

            # All other methods — same hook on all layers
            else:
                hook_fn       = cfg["hook"]
                hook_factories = [
                    (li, hook_fn(fault_type, fault_frac))
                    for li in ALL_LAYERS
                ]
                score = evaluate_with_hooks(
                    model, tokenizer, hook_factories, n_problems
                )

            all_scores[method_key].append(score)
            print(f"  {cfg['desc']:<30}: {score:.4f}")

    # ── Paper table ────────────────────────────────────────────────────────────
    m_clean  = np.mean(all_scores["1_clean"])
    m_faulty = np.mean(all_scores["2_faulty"])
    drop     = m_clean - m_faulty

    print("\n" + "="*75)
    print("COMPARISON TABLE — Established Fault Recovery Methods on LFM2.5")
    print("="*75)
    print(f"Fault: {fault_type}  fraction={fault_frac}  "
          f"n_seeds={n_seeds}  n_problems={n_problems}")
    print(f"\n{'Method':<30} {'Score':<20} {'Drop':>8} "
          f"{'Recovery':>10}  Citation")
    print("-"*75)

    rows = []
    for method_key, cfg in methods.items():
        scores = all_scores[method_key]
        mean   = np.mean(scores)
        std    = np.std(scores)
        ci     = 1.96 * std / np.sqrt(len(scores)) if len(scores) > 1 else 0
        d      = m_clean - mean
        rec    = (mean - m_faulty) / (drop + 1e-8) \
                 if method_key not in ("1_clean", "2_faulty") else None
        rec_str = f"{rec:+.1%}" if rec is not None else "—"

        print(f"  {cfg['desc']:<28} {mean:.4f}±{ci:.4f}  "
              f"{d:>8.4f}  {rec_str:>10}  {cfg.get('cite','')}")

        rows.append({
            "method":    cfg["desc"],
            "mean":      round(mean, 4),
            "ci_95":     round(ci, 4),
            "drop":      round(d, 4),
            "recovery":  round(rec, 4) if rec is not None else None,
            "cite":      cfg.get("cite", ""),
        })

    df = pd.DataFrame(rows)
    df.to_csv("/kaggle/working/method_comparison.csv", index=False)
    print("\nSaved → /kaggle/working/method_comparison.csv")

    # Best method
    recovery_rows = [r for r in rows if r["recovery"] is not None]
    best = max(recovery_rows, key=lambda x: x["recovery"] or 0)
    print(f"\nBest recovery: {best['method']} ({best['recovery']:.1%})")
    print(f"Cite: {best['cite']}")

    return df


Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [3]:
def build_layer_thresholds(model, tokenizer, n_texts=100):
    """
    Calibrates per-layer clip thresholds from CLEAN inference.
    Run this ONCE before fault injection.
    
    For each layer: threshold = mean(|activation|) + 3*std(|activation|)
    This is the maximum expected activation magnitude under normal operation.
    
    LFM2-specific: LIV layers have different activation scales than
    GQA layers due to the multiplicative gating. Calibrating separately
    gives better recovery than a fixed global threshold.
    
    This is your novel contribution:
    Clipper (established) + per-layer calibration + LFM2 layer awareness
    = Calibrated Clipper for LFM2 (CC-LFM2)
    """
    from datasets import load_dataset
    dataset  = load_dataset("wikitext", "wikitext-2-raw-v1",
                            split="train[:200]")
    captured = {i: [] for i in range(14)}

    def make_hook(li):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            captured[li].append(h.detach().abs().float())
        return hook

    hooks = [model.model.layers[li].register_forward_hook(make_hook(li))
             for li in range(14)]

    count = 0
    model.eval()
    with torch.no_grad():
        for sample in dataset:
            text = sample["text"].strip()
            if len(text) < 20: continue
            inputs = tokenizer(text, return_tensors="pt",
                               truncation=True, max_length=64).to(device)
            try: _ = model(**inputs)
            except: pass
            count += 1
            if count >= n_texts: break

    for h in hooks: h.remove()

    thresholds = {}
    print(f"\n{'Layer':>6} {'Type':<6} {'Mean|act|':>10} "
          f"{'Std|act|':>10} {'Threshold':>10}")
    print("-"*46)

    for li in range(14):
        if not captured[li]: continue
        all_acts = torch.cat([a.reshape(-1) for a in captured[li]])
        mean     = all_acts.mean().item()
        std      = all_acts.std().item()
        thresh   = mean + 3.0 * std   # 3σ above mean absolute value
        thresholds[li] = thresh
        ltype = "LIV" if li in LIV_LAYERS else "GQA"
        print(f"  {li:>4}  {ltype:<6}  {mean:>10.4f}  "
              f"{std:>10.4f}  {thresh:>10.4f}")

    return thresholds


def hook_cc_lfm2(layer_idx, thresholds,
                  fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    CC-LFM2: Calibrated Clipper for LFM2.
    
    Uses per-layer calibrated thresholds from clean inference.
    LIV layers get their own threshold, GQA layers get their own.
    
    Recovery: clip to [-threshold_i, +threshold_i] per layer i.
    All threshold values are from CLEAN model — not affected by fault.
    
    Cite as: CC-LFM2 (novel) combining
      Clipper principle (Hoang et al. DATE 2020) +
      Per-layer calibration from LFM2 architecture profiling
    """
    threshold = thresholds.get(layer_idx, 10.0)

    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        # Use calibrated per-layer threshold — not corrupted statistics
        h    = torch.nan_to_num(h, nan=0.0,
                                 posinf=threshold, neginf=-threshold)
        h    = torch.clamp(h, -threshold, threshold)
        return (h,) + rest if rest else h
    return hook


# # ── Build thresholds from clean model ─────────────────────────────────────────
# print("Building calibrated thresholds from clean model...")
# layer_thresholds = build_layer_thresholds(model, tokenizer, n_texts=100)

# ── Run final comparison: zero_fill vs clipper vs CC-LFM2 ─────────────────────
# N_SEEDS    = 5
# N_PROBLEMS = 100
# ALL_LAYERS = list(range(14))

# results = {"clean": [], "faulty": [], "zero_fill": [],
#            "clipper_fixed": [], "cc_lfm2": []}

# for seed in range(N_SEEDS):
#     print(f"\n── Seed {seed} ──")
#     random.seed(seed); torch.manual_seed(seed)

#     # Clean
#     lfm   = LFM2(model=model, tokenizer=tokenizer)
#     bench = IFEval(n_problems=N_PROBLEMS)
#     bench.evaluate(model=lfm)
#     results["clean"].append(bench.overall_score)

#     # Faulty
#     factories = [(li, hook_faulty()) for li in ALL_LAYERS]
#     results["faulty"].append(
#         evaluate_with_hooks(model, tokenizer, factories, N_PROBLEMS)
#     )

#     # Zero-fill
#     factories = [(li, hook_zero_fill()) for li in ALL_LAYERS]
#     results["zero_fill"].append(
#         evaluate_with_hooks(model, tokenizer, factories, N_PROBLEMS)
#     )

#     # Clipper fixed threshold=10
#     factories = [(li, hook_clipper(threshold=10.0)) for li in ALL_LAYERS]
#     results["clipper_fixed"].append(
#         evaluate_with_hooks(model, tokenizer, factories, N_PROBLEMS)
#     )

#     # CC-LFM2 — calibrated per layer
#     factories = [(li, hook_cc_lfm2(li, layer_thresholds))
#                  for li in ALL_LAYERS]
#     results["cc_lfm2"].append(
#         evaluate_with_hooks(model, tokenizer, factories, N_PROBLEMS)
#     )

#     print(f"  clean={results['clean'][-1]:.4f}  "
#           f"faulty={results['faulty'][-1]:.4f}  "
#           f"zero_fill={results['zero_fill'][-1]:.4f}  "
#           f"clipper={results['clipper_fixed'][-1]:.4f}  "
#           f"cc_lfm2={results['cc_lfm2'][-1]:.4f}")

## ── Final table ────────────────────────────────────────────────────────────────
# m_c = np.mean(results["clean"])
# m_f = np.mean(results["faulty"])
# drop = m_c - m_f

# print("\n" + "="*70)
# print("FINAL TABLE — CC-LFM2 vs Baselines")
# print("="*70)
# print(f"\n{'Method':<30} {'Score':<20} {'Recovery':>10}  Note")
# print("-"*70)

# for key, label, note in [
#     ("clean",        "Clean baseline",       "—"),
#     ("faulty",       "Faulty (no recovery)", "—"),
#     ("zero_fill",    "Zero-fill",            "generic — fixed value 0"),
#     ("clipper_fixed","Clipper (threshold=10)","generic — arbitrary threshold"),
#     ("cc_lfm2",      "CC-LFM2 (novel)",      "calibrated per LFM2 layer"),
# ]:
#     scores = results[key]
#     mean   = np.mean(scores)
#     ci     = 1.96 * np.std(scores) / np.sqrt(len(scores))
#     rec    = (mean - m_f) / (drop + 1e-8) \
#              if key not in ("clean","faulty") else None
#     rec_str = f"{rec:+.1%}" if rec is not None else "—"
#     print(f"  {label:<28} {mean:.4f}±{ci:.4f}  {rec_str:>10}  {note}")

# pd.DataFrame({k: results[k] for k in results}).to_csv(
#     "/kaggle/working/cc_lfm2_results.csv", index=False
# )
# print("\nSaved → /kaggle/working/cc_lfm2_results.csv")

In [4]:
# from typing import List
# import torch, copy, random, os, json
# import numpy as np
# import pandas as pd
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from deepeval.models.base_model import DeepEvalBaseLLM
# from deepeval.benchmarks import IFEval

# device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# MODEL_PATH  = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"
# RESULTS_CSV = "/kaggle/working/novel_recovery_results.csv"
# RAW_CSV     = "/kaggle/working/novel_recovery_raw_scores.csv"

# LIV_LAYERS  = [0, 1, 3, 5, 7, 9, 11, 13]
# GQA_LAYERS  = [2, 4, 6, 8, 10, 12]
# FAULT_FRAC  = 0.01

# # All fault models now exercised, not just "nan"/"inf" one-at-a-time.
# # bitflip_* mimics a single-event-upset (SEU) in on-chip SRAM/registers —
# # the standard soft-error model in the resilient-computing literature
# # (e.g. Ruospo et al. DATE 2023) — applied directly to the tensor's own
# # bit width (16-bit for bfloat16, 32-bit for float32).
# FAULT_TYPES = [
#     "nan",               # stuck-at-NaN               (existing)
#     "inf",               # stuck-at-+Inf               (existing)
#     "large",             # stuck-at-huge-finite-value   (existing)
#     "zero",               # stuck-at-zero               (new)
#     "sign_flip",          # sign bit corruption          (new)
#     "bitflip_mantissa",   # SEU in mantissa — small perturbation   (new)
#     "bitflip_exponent",   # SEU in exponent — magnitude blow-up    (new)
#     "bitflip_random",     # SEU anywhere in the word                (new)
# ]

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
# model     = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16
# ).to(device)
# model.eval()


# # ── LFM2 wrapper ──────────────────────────────────────────────────────────────
# class LFM2(DeepEvalBaseLLM):
#     def __init__(self, model, tokenizer):
#         self.model = model; self.tokenizer = tokenizer
#     def load_model(self): return self.model
#     def generate(self, prompt: str) -> str:
#         inputs = self.tokenizer([prompt], return_tensors="pt").to(device)
#         try:
#             ids = self.model.generate(
#                 **inputs, max_new_tokens=100,
#                 do_sample=False, temperature=None, top_p=None
#             )
#             return self.tokenizer.batch_decode(
#                 ids, skip_special_tokens=True
#             )[0]
#         except RuntimeError:
#             return ""
#     async def a_generate(self, prompt: str) -> str:
#         return self.generate(prompt)
#     def get_model_name(self): return "LFM2-230M"
#     def __call__(self, prompt: str) -> str: return self.generate(prompt)


# # ══════════════════════════════════════════════════════════════════════════════
# # FAULT INJECTION — now returns (h, mask). `mask` marks exactly which
# # elements were faulted, as if flagged by an ECC/parity-style hardware
# # detector (location known, correct value unknown — the standard
# # assumption behind zero-fill / TMR / Ranger-style recovery). This lets
# # every recovery method work identically regardless of fault model —
# # previously they detected corruption via isnan()/isinf(), which silently
# # missed "large", "zero", "sign_flip" and bitflip faults that don't
# # produce a NaN or an Inf.
# # ══════════════════════════════════════════════════════════════════════════════

# def _bit_layout(dtype):
#     """(total_bits, exponent_bits, mantissa_bits). fp32 and bf16 both use 8 exponent bits."""
#     nbits    = torch.finfo(dtype).bits
#     exp_bits = 8
#     return nbits, exp_bits, nbits - 1 - exp_bits


# def _bitflip(x, region="random"):
#     """Flip one random bit per element, native to the tensor's own bit width."""
#     nbits, exp_bits, mant_bits = _bit_layout(x.dtype)
#     int_dtype = {16: torch.int16, 32: torch.int32}[nbits]

#     if region == "sign":
#         lo, hi = nbits - 1, nbits - 1
#     elif region == "exponent":
#         lo, hi = mant_bits, nbits - 2
#     elif region == "mantissa":
#         lo, hi = 0, mant_bits - 1
#     else:  # "random" — anywhere in the word
#         lo, hi = 0, nbits - 1

#     int_view  = x.view(int_dtype)
#     bit_pos   = torch.randint(lo, hi + 1, x.shape, device=x.device, dtype=int_dtype)
#     flip_mask = torch.ones_like(bit_pos) << bit_pos
#     return (int_view ^ flip_mask).view(x.dtype)


# def _inject(h, fault_type, fault_frac):
#     """
#     Shared fault injection. Corrupts `fault_frac` of elements in `h`
#     according to `fault_type` and returns (h, mask) — mask is a boolean
#     tensor, same shape as h, True at every faulted position.
#     """
#     flat    = h.reshape(-1)
#     n       = max(1, int(len(flat) * fault_frac))
#     indices = torch.randperm(len(flat), device=flat.device)[:n]

#     if fault_type == "nan":
#         flat[indices] = float('nan')
#     elif fault_type == "inf":
#         flat[indices] = float('inf')
#     elif fault_type == "neg_inf":
#         flat[indices] = float('-inf')
#     elif fault_type == "large":
#         flat[indices] = torch.finfo(flat.dtype).max / 2
#     elif fault_type == "zero":
#         flat[indices] = 0.0
#     elif fault_type == "sign_flip":
#         flat[indices] = -flat[indices]
#     elif fault_type == "bitflip_random":
#         flat[indices] = _bitflip(flat[indices], region="random")
#     elif fault_type == "bitflip_exponent":
#         flat[indices] = _bitflip(flat[indices], region="exponent")
#     elif fault_type == "bitflip_mantissa":
#         flat[indices] = _bitflip(flat[indices], region="mantissa")
#     elif fault_type == "bitflip_sign":
#         flat[indices] = _bitflip(flat[indices], region="sign")
#     else:
#         raise ValueError(f"Unknown fault_type: {fault_type}")

#     mask = torch.zeros_like(flat, dtype=torch.bool)
#     mask[indices] = True
#     return h, mask.reshape(h.shape)


# # ── Evaluate with hooks ───────────────────────────────────────────────────────
# def evaluate_with_hooks(model, tokenizer, hook_factories, n_problems):
#     handles = []
#     for li, hook_fn in hook_factories:
#         h = model.model.layers[li].register_forward_hook(hook_fn)
#         handles.append(h)
#     lfm   = LFM2(model=model, tokenizer=tokenizer)
#     bench = IFEval(n_problems=n_problems)
#     bench.evaluate(model=lfm)
#     score = bench.overall_score
#     for h in handles:
#         h.remove()
#     return score


# # ══════════════════════════════════════════════════════════════════════════════
# # CALIBRATION — run once on clean model, fault-independent
# # ══════════════════════════════════════════════════════════════════════════════

# def calibrate_layer_means(model, tokenizer, n_texts=150):
#     """Records mean activation value per layer from clean inference (fixed, not re-estimated per fault)."""
#     dataset  = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:300]")
#     captured = {i: [] for i in range(14)}

#     def make_hook(li):
#         def hook(module, inp, out):
#             h = out[0] if isinstance(out, tuple) else out
#             captured[li].append(h.detach().float().mean().item())
#         return hook

#     hooks = [model.model.layers[li].register_forward_hook(make_hook(li))
#              for li in range(14)]

#     count = 0
#     model.eval()
#     with torch.no_grad():
#         for sample in dataset:
#             text = sample["text"].strip()
#             if len(text) < 20: continue
#             inputs = tokenizer(text, return_tensors="pt",
#                                truncation=True, max_length=64).to(device)
#             try: _ = model(**inputs)
#             except: pass
#             count += 1
#             if count >= n_texts: break

#     for h in hooks: h.remove()

#     means = {}
#     print(f"\n{'Layer':>6} {'Type':<6} {'Clean mean':>12}")
#     print("-"*28)
#     for li in range(14):
#         if captured[li]:
#             m = float(np.mean(captured[li]))
#             means[li] = m
#             ltype = "LIV" if li in LIV_LAYERS else "GQA"
#             print(f"  {li:>4}  {ltype:<6}  {m:>12.6f}")

#     return means


# # ══════════════════════════════════════════════════════════════════════════════
# # HOOK DEFINITIONS — all now use the `mask` returned by _inject, instead of
# # isnan()/isinf(), so they correctly detect and recover from EVERY fault type.
# # ══════════════════════════════════════════════════════════════════════════════

# def hook_faulty(fault_type, fault_frac=FAULT_FRAC):
#     """No recovery — fault only."""
#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, _ = _inject(h, fault_type, fault_frac)
#         return (h,) + rest if rest else h
#     return hook


# def hook_zero_fill(fault_type, fault_frac=FAULT_FRAC):
#     """Zero-fill: replace every faulted position with 0."""
#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, bad = _inject(h, fault_type, fault_frac)
#         h = torch.where(bad, torch.zeros_like(h), h)
#         return (h,) + rest if rest else h
#     return hook


# def hook_calibrated_mean(layer_idx, layer_means, fault_type, fault_frac=FAULT_FRAC):
#     """Idea 1: replace faulted positions with the pre-calibrated clean mean for this layer."""
#     fill_val = layer_means.get(layer_idx, 0.0)

#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, bad = _inject(h, fault_type, fault_frac)
#         fill = torch.full_like(h, fill_val)
#         h = torch.where(bad, fill, h)
#         return (h,) + rest if rest else h
#     return hook


# def hook_residual_preservation(fault_type, fault_frac=FAULT_FRAC):
#     """Idea 2: replace faulted output positions with the (clean) layer input — residual stream."""
#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, bad = _inject(h, fault_type, fault_frac)

#         if bad.any() and inp is not None and len(inp) > 0:
#             layer_input = inp[0] if isinstance(inp, tuple) else inp
#             if layer_input.shape == h.shape:
#                 h = torch.where(bad, layer_input.to(h.dtype), h)
#             else:
#                 h = torch.where(bad, torch.zeros_like(h), h)
#         return (h,) + rest if rest else h
#     return hook


# def hook_gate_gated(layer_idx, layer_means, fault_type, fault_frac=FAULT_FRAC):
#     """Idea 3: LIV layers get gate-magnitude-scaled fill; GQA layers get zero-fill."""
#     is_liv    = layer_idx in LIV_LAYERS
#     base_mean = layer_means.get(layer_idx, 0.0)

#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, bad = _inject(h, fault_type, fault_frac)

#         if not bad.any():
#             return (h,) + rest if rest else h

#         if is_liv:
#             valid = h[~bad]
#             if len(valid) > 100:
#                 current_mean = valid.abs().mean().item()
#                 calib_mean   = abs(base_mean) if abs(base_mean) > 1e-6 else 1.0
#                 gate_scale   = min(current_mean / calib_mean, 2.0)
#                 fill_val     = base_mean * gate_scale
#             else:
#                 fill_val = base_mean
#             h = torch.where(bad, torch.full_like(h, fill_val), h)
#         else:
#             h = torch.where(bad, torch.zeros_like(h), h)

#         return (h,) + rest if rest else h
#     return hook


# def hook_combination(layer_idx, layer_means, fault_type, fault_frac=FAULT_FRAC):
#     """Idea 4: LIV → residual preservation, GQA → calibrated mean-fill."""
#     is_liv   = layer_idx in LIV_LAYERS
#     fill_val = layer_means.get(layer_idx, 0.0)

#     def hook(module, inp, out):
#         h    = out[0] if isinstance(out, tuple) else out
#         rest = out[1:] if isinstance(out, tuple) else None
#         h, bad = _inject(h, fault_type, fault_frac)

#         if not bad.any():
#             return (h,) + rest if rest else h

#         if is_liv and inp is not None and len(inp) > 0:
#             layer_input = inp[0] if isinstance(inp, tuple) else inp
#             if layer_input.shape == h.shape:
#                 h = torch.where(bad, layer_input.to(h.dtype), h)
#             else:
#                 h = torch.where(bad, torch.full_like(h, fill_val), h)
#         else:
#             h = torch.where(bad, torch.full_like(h, fill_val), h)

#         return (h,) + rest if rest else h
#     return hook


# # ══════════════════════════════════════════════════════════════════════════════
# # MAIN EXPERIMENT — every method now evaluated under every fault type
# # ══════════════════════════════════════════════════════════════════════════════

# ALL_LAYERS = list(range(14))
# N_SEEDS    = 6     # reduce (e.g. 2) for a quick smoke test — 8 fault types x
# N_PROBLEMS = 100    # 6 methods x N_SEEDS runs of IFEval adds up fast
# INITIAL = 59

# # Step 1: Calibrate (fault-independent, run once)
# print("="*60)
# print("STEP 1: Calibrating layer means from clean model")
# print("="*60)
# layer_means = calibrate_layer_means(model, tokenizer, n_texts=150)

# # Step 2: Clean baseline — fault-independent, reused across all fault types
# print("\n" + "="*60)
# print("STEP 2: Clean baseline (shared across all fault types)")
# print("="*60)
# clean_scores = []
# for seed in range(INITIAL, N_SEEDS+INITIAL):
#     random.seed(seed); torch.manual_seed(seed)
#     lfm   = LFM2(model=model, tokenizer=tokenizer)
#     bench = IFEval(n_problems=N_PROBLEMS)
#     bench.evaluate(model=lfm)
#     clean_scores.append(bench.overall_score)
#     print(f"  seed {seed}: {bench.overall_score:.4f}")

# # method_hooks = {
# #     "faulty":     lambda li, ft: hook_faulty(ft, FAULT_FRAC),
# #     "zero_fill":  lambda li, ft: hook_zero_fill(ft, FAULT_FRAC),
# #     "cal_mean":   lambda li, ft: hook_calibrated_mean(li, layer_means, ft, FAULT_FRAC),
# #     "residual":   lambda li, ft: hook_residual_preservation(ft, FAULT_FRAC),
# #     "gate_gated": lambda li, ft: hook_gate_gated(li, layer_means, ft, FAULT_FRAC),
# #     "combo":      lambda li, ft: hook_combination(li, layer_means, ft, FAULT_FRAC),
# # }

# method_hooks = {
#     "faulty":     lambda li, ft: hook_faulty(ft, FAULT_FRAC),
#     "zero_fill":  lambda li, ft: hook_zero_fill(ft, FAULT_FRAC),
#     "residual":   lambda li, ft: hook_residual_preservation(ft, FAULT_FRAC),
#     "combo":      lambda li, ft: hook_combination(li, layer_means, ft, FAULT_FRAC),
# }




# # Step 3: every method x every fault type x every seed
# print("\n" + "="*60)
# print("STEP 3: Running all recovery methods across all fault types")
# print("="*60)

# results = {}  # {(fault_type, method): [scores per seed]}
# for fault_type in FAULT_TYPES:
#     print(f"\n{'#'*70}\n# FAULT TYPE: {fault_type}\n{'#'*70}")
#     for method_name in method_hooks:
#         results[(fault_type, method_name)] = []

#     for seed in range(INITIAL, N_SEEDS+INITIAL):
#         random.seed(seed); torch.manual_seed(seed)
#         print(f"\n── Seed {seed}/{N_SEEDS-1} [{fault_type}] ──")
#         for method_name, hook_factory in method_hooks.items():
#             factories = [(li, hook_factory(li, fault_type)) for li in ALL_LAYERS]
#             s = evaluate_with_hooks(model, tokenizer, factories, N_PROBLEMS)
#             results[(fault_type, method_name)].append(s)
#             print(f"  {method_name:<12}: {s:.4f}")

# # ── Results table ─────────────────────────────────────────────────────────────
# m_c = np.mean(clean_scores)

# # labels = {
# #     "faulty":     ("Faulty (no recovery)",   "—"),
# #     "zero_fill":  ("Zero-fill",              "generic baseline"),
# #     "cal_mean":   ("Idea 1: Cal. Mean-Fill", "per-layer calibration"),
# #     "residual":   ("Idea 2: Residual Pres.", "LFM2 residual stream"),
# #     "gate_gated": ("Idea 3: Gate-Gated",     "LIV gate magnitude"),
# #     "combo":      ("Idea 4: Combination",    "Idea2(LIV)+Idea1(GQA)"),
# # }

# labels = {
#     "faulty":     ("Faulty (no recovery)",   "—"),
#     "zero_fill":  ("Zero-fill",              "generic baseline"),
#     "residual":   ("Idea 2: Residual Pres.", "LFM2 residual stream"),
#     "combo":      ("Idea 4: Combination",    "Idea2(LIV)+Idea1(GQA)"),
# }

# rows = []
# for fault_type in FAULT_TYPES:
#     m_f  = np.mean(results[(fault_type, "faulty")])
#     drop = m_c - m_f

#     print("\n" + "="*80)
#     print(f"RESULTS TABLE — fault_type={fault_type}  frac={FAULT_FRAC}  "
#           f"n_seeds={N_SEEDS}  n_problems={N_PROBLEMS}")
#     print("="*80)
#     print(f"  {'Clean baseline':<28} {m_c:.4f}")
#     print(f"{'Method':<30} {'Score':<18} {'Drop':>8} {'Recovery':>10}  {'Note'}")
#     print("-"*80)

#     for method_name, (label, note) in labels.items():
#         s    = results[(fault_type, method_name)]
#         mean = np.mean(s)
#         std  = np.std(s)
#         ci   = 1.96 * std / np.sqrt(len(s)) if len(s) > 1 else 0.0
#         d    = m_c - mean
#         rec  = (mean - m_f) / (drop + 1e-8) if method_name != "faulty" else None
#         rec_str = f"{rec:+.1%}" if rec is not None else "—"

#         print(f"  {label:<28} {mean:.4f}±{ci:.4f}  {d:>8.4f}  {rec_str:>10}  {note}")

#         rows.append({
#             "fault_type": fault_type,
#             "method":     label,
#             "mean":       round(mean, 4),
#             "std":        round(std, 4),
#             "ci_95":      round(ci, 4),
#             "drop":       round(d, 4),
#             "recovery":   round(rec, 4) if rec is not None else None,
#             "note":       note,
#         })

#     novel = [r for r in rows if r["fault_type"] == fault_type
#              and r["method"] not in ("Faulty (no recovery)",) and r["recovery"] is not None]
#     if novel:
#         best = max(novel, key=lambda x: x["recovery"])
#         zf_rec = next(r["recovery"] for r in novel if "Zero" in r["method"])
#         print(f"\n  Best method for {fault_type}: {best['method']} "
#               f"({best['recovery']:+.1%} recovery)")
#         print(f"  Zero-fill baseline: {zf_rec:+.1%} recovery  "
#               f"(improvement: {best['recovery'] - zf_rec:+.1%})")

# # ── Save ─────────────────────────────────────────────────────────────────────
# df = pd.DataFrame(rows)
# df.to_csv(RESULTS_CSV, index=False)

# raw = {f"{ft}__{m}": results[(ft, m)] for ft in FAULT_TYPES for m in method_hooks}
# pd.DataFrame(raw).to_csv(RAW_CSV, index=False)

# print(f"\nSaved:")
# print(f"  {RESULTS_CSV}")
# print(f"  {RAW_CSV}")

# # ── Cross-fault-type summary: which method wins most often ───────────────────
# print("\n" + "="*80)
# print("CROSS-FAULT-TYPE SUMMARY — average recovery per method")
# print("="*80)
# summary = df[df["method"] != "Faulty (no recovery)"].groupby("method")["recovery"].mean()
# summary = summary.sort_values(ascending=False)
# for method, avg_rec in summary.items():
#     print(f"  {method:<28} avg recovery across {len(FAULT_TYPES)} fault types: {avg_rec:+.1%}")


In [5]:
import torch, random, os
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.benchmarks import IFEval
from typing import List

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16
).to(device)
model.eval()

FAULT_FRAC  = 0.01
INITIAL_SEED = 4
N_SEEDS     = 5
N_PROBLEMS  = 100
RESULTS_CSV = "/kaggle/working/related_work_raw_accuracy.csv"

FAULT_TYPES = [
    "nan", "inf", "large", "zero",
    "sign_flip", "bitflip_mantissa",
    "bitflip_exponent", "bitflip_random",
]


# ── LFM2 wrapper ──────────────────────────────────────────────────────────────
class LFM2(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model = model; self.tokenizer = tokenizer
    def load_model(self): return self.model
    def generate(self, prompt: str) -> str:
        inputs = self.tokenizer([prompt], return_tensors="pt").to(device)
        try:
            ids = self.model.generate(
                **inputs, max_new_tokens=100,
                do_sample=False, temperature=None, top_p=None
            )
            return self.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
        except RuntimeError:
            return ""
    async def a_generate(self, prompt: str) -> str: return self.generate(prompt)
    def get_model_name(self): return "LFM2-230M"
    def __call__(self, prompt: str) -> str: return self.generate(prompt)


# ── Fault injection (same as your existing code) ──────────────────────────────
def _bitflip(x, region="random"):
    nbits     = 16   # bfloat16
    int_dtype = torch.int16
    if region == "sign":       lo, hi = 15, 15
    elif region == "exponent": lo, hi = 7, 14
    elif region == "mantissa": lo, hi = 0, 6
    else:                      lo, hi = 0, 15
    int_view  = x.view(int_dtype)
    bit_pos   = torch.randint(lo, hi+1, x.shape, device=x.device, dtype=int_dtype)
    flip_mask = torch.ones_like(bit_pos) << bit_pos
    return (int_view ^ flip_mask).view(x.dtype)


def _inject(h, fault_type, fault_frac):
    flat    = h.reshape(-1)
    n       = max(1, int(len(flat) * fault_frac))
    indices = torch.randperm(len(flat), device=flat.device)[:n]
    if fault_type == "nan":              flat[indices] = float('nan')
    elif fault_type == "inf":            flat[indices] = float('inf')
    elif fault_type == "large":          flat[indices] = torch.finfo(flat.dtype).max / 2
    elif fault_type == "zero":           flat[indices] = 0.0
    elif fault_type == "sign_flip":      flat[indices] = -flat[indices]
    elif fault_type == "bitflip_random":   flat[indices] = _bitflip(flat[indices], "random")
    elif fault_type == "bitflip_exponent": flat[indices] = _bitflip(flat[indices], "exponent")
    elif fault_type == "bitflip_mantissa": flat[indices] = _bitflip(flat[indices], "mantissa")
    mask = torch.zeros_like(flat, dtype=torch.bool)
    mask[indices] = True
    return h, mask.reshape(h.shape)


# ── Three method hooks ────────────────────────────────────────────────────────

def hook_ranger(fault_type, fault_frac=FAULT_FRAC):
    """
    Ranger — Wandel et al. DATE 2021.
    Clips to mean ± 6σ computed from non-faulted elements.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h, bad = _inject(h, fault_type, fault_frac)
        valid  = h[~bad]
        if len(valid) > 0:
            mu  = valid.mean()
            sig = valid.std()
            lo  = mu - 6 * sig
            hi  = mu + 6 * sig
            fill = torch.where(h > 0, hi, lo)
            h = torch.where(bad, fill, h)
            h = torch.clamp(h, lo, hi)
        else:
            h = torch.zeros_like(h)
        return (h,) + rest if rest else h
    return hook


def hook_clipper(fault_type, fault_frac=FAULT_FRAC, threshold=10.0):
    """
    Clipper — Hoang et al. DATE 2020.
    Hard clips to [-threshold, +threshold], fills NaN with 0.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h, bad = _inject(h, fault_type, fault_frac)
        h = torch.where(bad, torch.zeros_like(h), h)
        h = torch.clamp(h, -threshold, threshold)
        return (h,) + rest if rest else h
    return hook


def hook_fmapavg(fault_type, fault_frac=FAULT_FRAC):
    """
    FmapAvg — Ruospo et al. DATE 2023.
    Replaces faulted positions with mean of non-faulted token positions.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h, bad = _inject(h, fault_type, fault_frac)
        if bad.any():
            good   = (~bad).float()
            h_safe = torch.where(bad, torch.zeros_like(h), h)
            mean   = h_safe.sum(dim=1, keepdim=True) / \
                     (good.sum(dim=1, keepdim=True) + 1e-8)
            h = torch.where(bad, mean.expand_as(h), h)
        return (h,) + rest if rest else h
    return hook


# ── Evaluate with hooks ───────────────────────────────────────────────────────
def evaluate_with_hooks(model, tokenizer, hook_factories, n_problems):
    handles = [
        model.model.layers[li].register_forward_hook(fn)
        for li, fn in hook_factories
    ]
    lfm   = LFM2(model=model, tokenizer=tokenizer)
    bench = IFEval(n_problems=n_problems)
    bench.evaluate(model=lfm)
    score = bench.overall_score
    for h in handles:
        h.remove()
    return score


# ── Main experiment ───────────────────────────────────────────────────────────
ALL_LAYERS = list(range(14))

methods = {
    "clipper":  lambda li, ft: hook_clipper(ft),
    "ranger":   lambda li, ft: hook_ranger(ft),
    "fmapavg":  lambda li, ft: hook_fmapavg(ft),
}

# raw_scores[fault_type][method] = [s0, s1, s2, ...]
raw_scores = {ft: {m: [] for m in methods} for ft in FAULT_TYPES}

for fault_type in FAULT_TYPES:
    print(f"\n{'='*55}")
    print(f"Fault type: {fault_type}")
    print(f"{'='*55}")

    for seed in range(INITIAL_SEED, INITIAL_SEED + N_SEEDS):
        random.seed(seed); torch.manual_seed(seed)
        print(f"  Seed {seed}:", end=" ", flush=True)

        for method_name, hook_factory in methods.items():
            factories = [(li, hook_factory(li, fault_type))
                         for li in ALL_LAYERS]
            score = evaluate_with_hooks(
                model, tokenizer, factories, N_PROBLEMS
            )
            raw_scores[fault_type][method_name].append(score)
            print(f"{method_name}={score:.4f}", end="  ", flush=True)

        print()

    # Save checkpoint after each fault type — crash safety
    rows = []
    for ft in FAULT_TYPES:
        for m in methods:
            scores = raw_scores[ft][m]
            for seed_i, s in enumerate(scores):
                rows.append({
                    "fault_type": ft,
                    "method":     m,
                    "seed":       seed_i,
                    "accuracy":   s,
                })
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    print(f"  Checkpoint saved → {RESULTS_CSV}")

print(f"\nFinal saved → {RESULTS_CSV}")
print("Done.")

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]


Fault type: nan
  Seed 4: 

README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

...

Overall IFEval Accuracy: 0.6700
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc


Processing 100 IFEval problems: 100%|██████████| 100/100 [03:12<00:00,  1.92s/it]

Overall IFEval Accuracy: 0.6000
Instruction 'punctuation:no_comma' Accuracy: 0.1667
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.2857
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.0000
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc


Processing 100 IFEval problems: 100%|██████████| 100/100 [00:07<00:00, 13.62it/s]

Overall IFEval Accuracy: 0.5000
Instruction 'punctuation:no_comma' Accuracy: 1.0000
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.0000
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.0000
Instruction 'combination:repeat_prompt' Accuracy: 0.0000
Instruction 'detectable_format:title' Accuracy: 0.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc